# Word Sense Disambiguation (WSD) Project
## TASK 1-4: Complete Implementation

This notebook implements:
1. Dataset preparation with ambiguous words
2. Lesk Algorithm (NLTK-based)
3. Embedding-based WSD (Sentence-BERT)
4. Comparative analysis and visualization

## Install and Import Required Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install nltk sentence-transformers scikit-learn pandas matplotlib

import nltk
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("✓ All libraries imported successfully!")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TASK 1: Dataset Preparation
### Create queries with ambiguous words

In [ ]:
# Create a list of queries with ambiguous words
queries_data = [
    {"query": "I need to book a flight to Paris", "ambiguous_word": "book"},
    {"query": "She placed the book on the shelf", "ambiguous_word": "book"},
    {"query": "The cricket bat is made of willow wood", "ambiguous_word": "bat"},
    {"query": "A bat flew out of the cave at night", "ambiguous_word": "bat"},
    {"query": "I need to deposit money at the bank", "ambiguous_word": "bank"},
    {"query": "We sat by the river bank and watched the sunset", "ambiguous_word": "bank"},
    {"query": "Please turn on the light in the room", "ambiguous_word": "light"},
    {"query": "The box is very light to carry", "ambiguous_word": "light"},
    {"query": "The football match was very exciting", "ambiguous_word": "match"},
    {"query": "Strike a match to light the candle", "ambiguous_word": "match"},
    {"query": "He joined the golf club last year", "ambiguous_word": "club"},
    {"query": "The security guard carried a club", "ambiguous_word": "club"},
    {"query": "How much do you charge for this service", "ambiguous_word": "charge"},
    {"query": "The battery needs to be charged overnight", "ambiguous_word": "charge"},
    {"query": "The official seal was stamped on the document", "ambiguous_word": "seal"},
    {"query": "We saw a seal swimming in the ocean", "ambiguous_word": "seal"},
    {"query": "The current situation is very challenging", "ambiguous_word": "current"},
    {"query": "The river current is very strong today", "ambiguous_word": "current"},
]

print(f"Created {len(queries_data)} queries with ambiguous words")
print("\nSample queries:")
for i, q in enumerate(queries_data[:3], 1):
    print(f"{i}. '{q['query']}' - ambiguous word: '{q['ambiguous_word']}'")

### Build Sense Dictionary using WordNet

In [ ]:
def build_sense_dictionary(word):
    """Build a sense dictionary for a given word using WordNet"""
    synsets = wn.synsets(word)
    sense_dict = []
    
    for synset in synsets:
        sense_info = {
            'synset_id': synset.name(),
            'definition': synset.definition(),
            'examples': synset.examples(),
            'pos': synset.pos()  # Part of speech
        }
        sense_dict.append(sense_info)
    
    return sense_dict

# Build sense dictionary for all ambiguous words
sense_dictionary = {}
ambiguous_words = list(set([q['ambiguous_word'] for q in queries_data]))

for word in ambiguous_words:
    sense_dictionary[word] = build_sense_dictionary(word)
    print(f"'{word}' has {len(sense_dictionary[word])} senses in WordNet")

# Display example
print(f"\n{'='*60}")
print(f"Example: Sense dictionary for 'book'")
print(f"{'='*60}")
for i, sense in enumerate(sense_dictionary['book'][:3], 1):
    print(f"\nSense {i}: {sense['synset_id']}")
    print(f"  Definition: {sense['definition']}")
    print(f"  Examples: {sense['examples']}")

### Prepare Dataset and Save as CSV

In [ ]:
# Prepare dataset with possible senses
dataset = []
for q_data in queries_data:
    query = q_data['query']
    ambiguous_word = q_data['ambiguous_word']
    possible_senses = sense_dictionary[ambiguous_word]
    
    dataset.append({
        'query': query,
        'ambiguous_word': ambiguous_word,
        'possible_senses': json.dumps(possible_senses)  # Convert to JSON string for CSV
    })

# Create DataFrame and save to CSV
df_dataset = pd.DataFrame(dataset)
df_dataset.to_csv('wsd_dataset.csv', index=False)
print(f"✓ Dataset saved to 'wsd_dataset.csv' with {len(df_dataset)} queries")
print(f"\nDataset preview:")
print(df_dataset[['query', 'ambiguous_word']].head())

In [ ]:
# Save sense dictionary as JSON
with open('sense_dictionary.json', 'w') as f:
    json.dump(sense_dictionary, f, indent=2)

print("✓ Sense dictionary saved to 'sense_dictionary.json'")
print(f"Total ambiguous words: {len(sense_dictionary)}")

## TASK 2: Implement WSD Algorithms
### 2a) Lesk Algorithm (NLTK)

In [ ]:
def lesk_wsd(query, ambiguous_word):
    """
    Perform WSD using Lesk algorithm
    
    Args:
        query: The sentence containing the ambiguous word
        ambiguous_word: The word to disambiguate
    
    Returns:
        Dictionary with synset, definition, synset_id, and status
    """
    # Tokenize the query
    tokens = word_tokenize(query.lower())
    
    # Use Lesk algorithm
    best_synset = lesk(tokens, ambiguous_word)
    
    if best_synset is None:
        return {
            'synset': None,
            'synset_id': None,
            'definition': 'No sense found',
            'status': 'Failed'
        }
    
    return {
        'synset': best_synset,
        'synset_id': best_synset.name(),
        'definition': best_synset.definition(),
        'status': 'Success'
    }

# Test Lesk algorithm on a sample query
sample_query = "I need to book a flight to Paris"
sample_word = "book"
result = lesk_wsd(sample_query, sample_word)

print(f"Query: '{sample_query}'")
print(f"Ambiguous word: '{sample_word}'")
print(f"\nLesk Result:")
print(f"  Synset ID: {result['synset_id']}")
print(f"  Definition: {result['definition']}")
print(f"  Status: {result['status']}")

### 2b) Embedding-based WSD (Sentence-BERT)

In [ ]:
# Load Sentence-BERT model
print("Loading Sentence-BERT model...")
sbert_model = SentenceTransformer('all-mpnet-base-v2')
print("✓ Model loaded successfully!")

In [ ]:
def embedding_wsd(query, ambiguous_word, sense_dict):
    """
    Perform WSD using Sentence-BERT embeddings
    
    Args:
        query: The sentence containing the ambiguous word
        ambiguous_word: The word to disambiguate
        sense_dict: Dictionary of possible senses for the word
    
    Returns:
        Dictionary with best synset, definition, similarity scores
    """
    # Get all senses for the ambiguous word
    senses = sense_dict.get(ambiguous_word, [])
    
    if not senses:
        return {
            'synset_id': None,
            'definition': 'No senses available',
            'similarity_scores': {},
            'best_score': 0.0,
            'status': 'Failed'
        }
    
    # Encode the query
    query_embedding = sbert_model.encode([query])[0]
    
    # Compute similarities for each sense
    similarity_scores = {}
    best_sense = None
    best_score = -1
    
    for sense in senses:
        # Combine definition and examples for better context
        sense_text = sense['definition']
        if sense['examples']:
            sense_text += " " + " ".join(sense['examples'])
        
        # Encode the sense
        sense_embedding = sbert_model.encode([sense_text])[0]
        
        # Compute cosine similarity
        similarity = cosine_similarity(
            query_embedding.reshape(1, -1),
            sense_embedding.reshape(1, -1)
        )[0][0]
        
        similarity_scores[sense['synset_id']] = float(similarity)
        
        if similarity > best_score:
            best_score = similarity
            best_sense = sense
    
    return {
        'synset_id': best_sense['synset_id'],
        'definition': best_sense['definition'],
        'similarity_scores': similarity_scores,
        'best_score': float(best_score),
        'status': 'Success'
    }

# Test embedding-based WSD on a sample query
sample_query = "I need to book a flight to Paris"
sample_word = "book"
result = embedding_wsd(sample_query, sample_word, sense_dictionary)

print(f"Query: '{sample_query}'")
print(f"Ambiguous word: '{sample_word}'")
print(f"\nEmbedding-based WSD Result:")
print(f"  Best Synset ID: {result['synset_id']}")
print(f"  Definition: {result['definition']}")
print(f"  Similarity Score: {result['best_score']:.4f}")
print(f"\n  All similarity scores:")
for synset_id, score in list(result['similarity_scores'].items())[:5]:
    print(f"    {synset_id}: {score:.4f}")

## TASK 3: Experimentation & Comparison
### Run both algorithms on all queries

In [ ]:
# Run both algorithms on all queries
results = []

print("Running WSD algorithms on all queries...")
print("="*70)

for i, q_data in enumerate(queries_data, 1):
    query = q_data['query']
    ambiguous_word = q_data['ambiguous_word']
    
    # Run Lesk algorithm
    lesk_result = lesk_wsd(query, ambiguous_word)
    
    # Run Embedding-based WSD
    embedding_result = embedding_wsd(query, ambiguous_word, sense_dictionary)
    
    # Check agreement
    agreement = (lesk_result['synset_id'] == embedding_result['synset_id']) if lesk_result['synset_id'] and embedding_result['synset_id'] else False
    
    result_entry = {
        'query': query,
        'ambiguous_word': ambiguous_word,
        'lesk_synset': lesk_result['synset_id'],
        'lesk_definition': lesk_result['definition'],
        'embedding_synset': embedding_result['synset_id'],
        'embedding_definition': embedding_result['definition'],
        'per_sense_similarity_scores': embedding_result['similarity_scores'],
        'best_similarity_score': embedding_result['best_score'],
        'agreement': agreement
    }
    
    results.append(result_entry)
    
    # Display progress
    if i % 5 == 0:
        print(f"Processed {i}/{len(queries_data)} queries...")

print(f"\n✓ Completed processing all {len(results)} queries!")

### Display Sample Results

In [ ]:
# Display sample results
print("="*80)
print("SAMPLE RESULTS (First 5 queries)")
print("="*80)

for i, result in enumerate(results[:5], 1):
    print(f"\n{i}. Query: '{result['query']}'")
    print(f"   Ambiguous word: '{result['ambiguous_word']}'")
    print(f"\n   Lesk Algorithm:")
    print(f"     Synset: {result['lesk_synset']}")
    print(f"     Definition: {result['lesk_definition']}")
    print(f"\n   Embedding-based:")
    print(f"     Synset: {result['embedding_synset']}")
    print(f"     Definition: {result['embedding_definition']}")
    print(f"     Best Score: {result['best_similarity_score']:.4f}")
    print(f"\n   Agreement: {'✓ Yes' if result['agreement'] else '✗ No'}")
    print("-"*80)

### Plot Similarity Scores (Bar Charts)

In [ ]:
# Plot similarity scores for selected queries
def plot_similarity_scores(results, num_plots=6):
    """Plot bar charts for similarity scores"""
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx in range(min(num_plots, len(results))):
        result = results[idx]
        scores = result['per_sense_similarity_scores']
        
        # Limit to top 5 senses for clarity
        sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]
        synsets = [s[0] for s in sorted_scores]
        similarities = [s[1] for s in sorted_scores]
        
        # Create bar chart
        ax = axes[idx]
        bars = ax.bar(range(len(synsets)), similarities, color='steelblue')
        
        # Highlight the best sense
        bars[0].set_color('darkgreen')
        
        ax.set_xticks(range(len(synsets)))
        ax.set_xticklabels([s.split('.')[0][:8] for s in synsets], rotation=45, ha='right')
        ax.set_ylabel('Cosine Similarity')
        ax.set_ylim([0, 1])
        ax.set_title(f"Query {idx+1}: '{result['ambiguous_word']}'", fontsize=10, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('similarity_scores_plots.png', dpi=300, bbox_inches='tight')
    print("✓ Similarity score plots saved as 'similarity_scores_plots.png'")
    plt.show()

plot_similarity_scores(results)

### Save Results to CSV and JSON

In [ ]:
# Prepare results for CSV (convert dict to JSON string)
results_for_csv = []
for result in results:
    result_copy = result.copy()
    result_copy['per_sense_similarity_scores'] = json.dumps(result['per_sense_similarity_scores'])
    results_for_csv.append(result_copy)

# Save to CSV
df_results = pd.DataFrame(results_for_csv)
df_results.to_csv('wsd_results.csv', index=False)
print("✓ Results saved to 'wsd_results.csv'")

# Save full results to JSON (with dict preserved)
with open('wsd_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✓ Results saved to 'wsd_results.json'")

## TASK 4: Summary Report
### Compute Statistics and Analysis

In [ ]:
# Calculate statistics
total_queries = len(results)
agreements = sum([1 for r in results if r['agreement']])
disagreements = total_queries - agreements
agreement_rate = (agreements / total_queries) * 100

print("="*80)
print("SUMMARY REPORT - Word Sense Disambiguation")
print("="*80)
print(f"\n1. OVERALL STATISTICS")
print(f"   • Total queries processed: {total_queries}")
print(f"   • Agreements: {agreements}")
print(f"   • Disagreements: {disagreements}")
print(f"   • Agreement rate: {agreement_rate:.2f}%")

# Count by ambiguous word
word_stats = {}
for result in results:
    word = result['ambiguous_word']
    if word not in word_stats:
        word_stats[word] = {'total': 0, 'agreements': 0}
    word_stats[word]['total'] += 1
    if result['agreement']:
        word_stats[word]['agreements'] += 1

print(f"\n2. AGREEMENT BY AMBIGUOUS WORD")
for word, stats in sorted(word_stats.items()):
    rate = (stats['agreements'] / stats['total']) * 100
    print(f"   • {word}: {stats['agreements']}/{stats['total']} ({rate:.1f}%)")

In [ ]:
# Display cases where methods differ
print(f"\n3. CASES WHERE METHODS DIFFER")
print("-"*80)

disagreement_cases = [r for r in results if not r['agreement']]

for i, result in enumerate(disagreement_cases, 1):
    print(f"\n{i}. Query: '{result['query']}'")
    print(f"   Ambiguous word: '{result['ambiguous_word']}'")
    print(f"   Lesk: {result['lesk_synset']}")
    print(f"   Embedding: {result['embedding_synset']}")
    print(f"   Lesk Definition: {result['lesk_definition'][:80]}...")
    print(f"   Embedding Definition: {result['embedding_definition'][:80]}...")
    print(f"   Similarity Score: {result['best_similarity_score']:.4f}")

In [ ]:
print(f"\n4. OBSERVATIONS ABOUT DIFFERENCES")
print("-"*80)

print("""
WHY DIFFERENCES OCCUR:

1. **Algorithm Methodology:**
   - Lesk Algorithm: Uses overlap between context words and dictionary definitions/examples.
     It relies on exact word matches and may miss semantic similarities.
   
   - Embedding-based: Uses dense vector representations to capture semantic meaning.
     It can understand context even without exact word matches.

2. **Context Understanding:**
   - Lesk: Limited to bag-of-words overlap, may miss subtle contextual cues
   - Embeddings: Better at capturing contextual nuances and semantic relationships

3. **Definition Quality:**
   - Lesk: Highly dependent on the quality and coverage of WordNet definitions
   - Embeddings: Can leverage pre-trained knowledge from large corpora

4. **Word Order:**
   - Lesk: Ignores word order (bag-of-words approach)
   - Embeddings: Preserves sequential information and sentence structure
""")

print(f"\n5. STRENGTHS AND WEAKNESSES")
print("-"*80)

print("""
LESK ALGORITHM:
Strengths:
  ✓ Simple and interpretable
  ✓ Fast computation
  ✓ No need for pre-trained models
  ✓ Works well when context words directly match definition words

Weaknesses:
  ✗ Relies on exact word overlap
  ✗ Misses semantic similarities (synonyms, paraphrases)
  ✗ Limited by WordNet definition quality
  ✗ Sensitive to context length

EMBEDDING-BASED WSD:
Strengths:
  ✓ Captures semantic similarities
  ✓ Better at understanding contextual nuances
  ✓ Leverages large-scale pre-trained knowledge
  ✓ Robust to paraphrasing and synonyms

Weaknesses:
  ✗ Requires pre-trained models (computational overhead)
  ✗ Less interpretable than Lesk
  ✗ May be overkill for simple cases
  ✗ Dependent on quality of embeddings
""")

## Summary of Generated Files

All outputs have been successfully generated:
- `wsd_dataset.csv` - Dataset with queries and ambiguous words
- `sense_dictionary.json` - Sense dictionary from WordNet
- `wsd_results.csv` - Comparison results in CSV format
- `wsd_results.json` - Detailed results in JSON format  
- `similarity_scores_plots.png` - Visualization of similarity scores
- This notebook - Complete implementation of all tasks

**To run the notebook:** Execute all cells sequentially from top to bottom.